# Example: Estimating Single Index Model Parameters with Bootstrap Uncertainty

In this example, we load the frozen synthetic training dataset (20 years of JumpHMM-generated market data), estimate Single Index Model (SIM) parameters for a demonstration ticker, quantify parameter uncertainty via bootstrap resampling, and then estimate parameters for the full 424-ticker universe and save the results.

> __Learning Objectives:__
>
> * __Estimate SIM parameters for a single asset:__ Regress asset returns on market returns via regularized ordinary least squares (OLS) to obtain alpha, beta, and residual volatility. Interpret the estimated parameters and evaluate the regression fit.
> * __Bootstrap uncertainty quantification:__ Resample residuals to build the empirical sampling distribution of alpha and beta. Compare bootstrap confidence intervals to theoretical standard errors.
> * __Estimate SIM parameters across the full ticker universe:__ Run SIM estimation and bootstrap for all tickers in the synthetic training dataset. Save the complete parameter set to disk for use in downstream sessions.

Let's get started!
___

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via the `Include.jl` file. This activates the local Julia environment and loads all dependencies.

In [ ]:
# --- Load packages and helper functions ---
include("Include.jl"); # The Include.jl file activates the local Julia environment and imports all dependencies.

Next, we'll load the 20-year synthetic training dataset using [the `MySyntheticTrainingDataSet()` function](../../code/docs/build/session1.html) and extract market growth rates (scaled log returns) and the full ticker list. We also define a `demo_ticker` variable that controls which ticker is used for the single-asset demonstrations in Tasks 1 and 2; change this to any ticker in the dataset to explore different assets.

The code below loads the dataset, extracts the market returns, and prints a summary.

In [ ]:
# --- Step 1: Load the frozen synthetic training dataset ---
training_data = MySyntheticTrainingDataSet();
dataset = training_data["dataset"];             # Dict of ticker DataFrames
all_tickers = training_data["tickers"];          # sorted vector of 424 ticker symbols
market_prices = training_data["market_prices"];  # market index price series
# drop the unused first observation so market_returns aligns with prices-derived growth rates
market_returns = training_data["market_returns"][2:end]; # market excess growth rates
T = length(market_returns);                       # number of daily return observations
Δt = 1.0 / 252.0;                                # daily time step (fraction of year)

# --- Step 2: Choose the demonstration ticker ---
# Change this to any ticker in all_tickers to explore a different asset.
demo_ticker = "NVDA";

# --- Step 3: Compute daily log growth rates for the demo ticker ---
demo_prices = dataset[demo_ticker][:, :close];
demo_returns = [(1.0/Δt) * log(demo_prices[t]/demo_prices[t-1]) for t ∈ 2:length(demo_prices)];

# --- Step 4: Print summary ---
println("Training data: $(T) daily returns, $(length(all_tickers)) tickers")
println("Demo ticker: $(demo_ticker)")
println("Market return observations: $(T)")
println("Demo ticker return observations: $(length(demo_returns))")

### Implementation
We define a helper function used in Task 1 to visualize the SIM regression fit.

In [ ]:
"""
    plot_sim_fit(ticker, est, actual_returns, market_rets) -> Plots.Plot

Generate a scatter plot of SIM-predicted vs. actual daily log growth rates for `ticker`.
Uses the point estimate `est::MySIMParameterEstimate` and the provided return vectors.
Returns a `Plots.Plot` object with predicted returns on the x-axis, actual returns on the y-axis,
and a dashed x = y reference line.
"""
function plot_sim_fit(ticker::String, est::MySIMParameterEstimate, 
        actual_returns::Vector{Float64}, market_rets::Vector{Float64})

    ŷ = est.α .+ est.β .* market_rets;  # SIM-predicted returns
    lims = (min(minimum(ŷ), minimum(actual_returns)), max(maximum(ŷ), maximum(actual_returns)));
    scatter(ŷ, actual_returns, alpha=0.3, ms=2, color=:navy, label="",
        xlabel="Predicted gᵢ", ylabel="Actual gᵢ",
        title="$(ticker) (R²=$(round(est.r², digits=3)), β=$(round(est.β, digits=2)))");
    plot!([lims...], [lims...], lw=2, ls=:dash, color=:red, label="x = y")
end;

___
## Task 1: Estimate SIM Parameters for a Single Ticker
We estimate the Single Index Model parameters $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ for the demo ticker via regularized OLS regression.

> __What are we going to do?__
>
> We regress the demo ticker's daily growth rates on market returns using [the `estimate_sim(...)` function](../../code/docs/build/session1.html). The result is a `MySIMParameterEstimate` containing Jensen's alpha, market beta, residual volatility, and the regression $R^{2}$.

A high-beta ticker like NVDA ($\beta > 1$) amplifies market moves, while a defensive ticker like JNJ ($\beta < 1$) dampens them. The $R^{2}$ measures what fraction of the asset's variance is explained by the market factor alone.

The code below calls `estimate_sim` and displays the results.

In [ ]:
# --- Step 1: Estimate SIM parameters for the demo ticker ---
demo_estimate = estimate_sim(market_returns, demo_returns, demo_ticker; δ=0.0, Δt=Δt);

# --- Step 2: Build a parameter summary DataFrame ---
params_df = DataFrame(
    "Parameter" => ["α (Jensen's alpha)", "β (market beta)", "σ_ε (residual vol)", "R²"],
    "Value" => [
        round(demo_estimate.α, digits=6),
        round(demo_estimate.β, digits=4),
        round(demo_estimate.σ_ε, digits=6),
        round(demo_estimate.r², digits=4)
    ],
    "Annualized" => [
        "$(round(demo_estimate.α * 252 * 10000, digits=1)) bps",
        "",
        "$(round(demo_estimate.σ_ε * sqrt(252) * 100, digits=1))%",
        ""
    ]
);

# --- Step 3: Display ---
println("SIM Parameter Estimates for $(demo_ticker):")
pretty_table(params_df; backend = :text,
    fit_table_in_display_horizontally = false,
    fit_table_in_display_vertically = false,
    table_format = TextTableFormat(borders = text_table_borders__compact))

> __Interpreting the SIM parameters:__
>
> A positive $\alpha$ means the asset generates excess return beyond what the market factor explains. A $\beta$ greater than 1 means the asset amplifies market moves (both up and down). The residual volatility $\sigma_{\varepsilon}$ captures idiosyncratic risk not explained by the market, and $R^{2}$ measures the fraction of total variance attributable to the market factor.

The scatter plot below shows predicted vs. actual daily growth rates for the demo ticker.

In [ ]:
# --- Scatter plot: SIM-predicted vs actual returns ---
plot_sim_fit(demo_ticker, demo_estimate, demo_returns, market_returns) |>
    p -> plot!(p, size=(600, 500))

___
## Task 2: Bootstrap Uncertainty Quantification
A point estimate of $\alpha$ and $\beta$ is incomplete without knowing how uncertain those estimates are. We use residual resampling bootstrap to build the empirical sampling distribution and compare it to the theoretical standard errors.

> __What are we going to do?__
>
> We call [the `bootstrap_sim(...)` function](../../code/docs/build/session1.html) with 1,000 replicates to generate the bootstrap distribution of $\hat{\alpha}$ and $\hat{\beta}$. We then compare the bootstrap 95% confidence intervals and standard errors to the theoretical (OLS) predictions.

If the Gaussian error assumption is reasonable, the bootstrap and theoretical standard errors should be close (ratio near 1.0). Wide confidence intervals indicate that the parameter estimate is unreliable.

The code below runs the bootstrap and prints the comparison.

In [ ]:
# --- Step 1: Bootstrap the demo ticker with 1,000 replicates ---
demo_bootstrap = bootstrap_sim(market_returns, demo_returns, demo_ticker;
    δ=0.0, Δt=Δt, n_bootstrap=1000, seed=42);

# --- Step 2: Extract results ---
est = demo_bootstrap["point_estimate"];

# --- Step 3: Build comparison DataFrame ---
bootstrap_df = DataFrame(
    "Parameter" => ["α", "β"],
    "Point Estimate" => [round(est.α, digits=6), round(est.β, digits=4)],
    "Bootstrap Mean" => [round(demo_bootstrap["alpha_mean"], digits=6), round(demo_bootstrap["beta_mean"], digits=4)],
    "95% CI Low" => [round(demo_bootstrap["alpha_ci_95"][1], digits=6), round(demo_bootstrap["beta_ci_95"][1], digits=4)],
    "95% CI High" => [round(demo_bootstrap["alpha_ci_95"][2], digits=6), round(demo_bootstrap["beta_ci_95"][2], digits=4)],
    "Bootstrap SE" => [round(demo_bootstrap["alpha_std"], digits=5), round(demo_bootstrap["beta_std"], digits=5)],
    "Theoretical SE" => [round(demo_bootstrap["theoretical_se"][1], digits=5), round(demo_bootstrap["theoretical_se"][2], digits=5)],
    "SE Ratio" => [
        round(demo_bootstrap["alpha_std"] / demo_bootstrap["theoretical_se"][1], digits=3),
        round(demo_bootstrap["beta_std"] / demo_bootstrap["theoretical_se"][2], digits=3)
    ]
);

# --- Step 4: Display ---
println("$(demo_ticker) Bootstrap Results (1,000 replicates):")
pretty_table(bootstrap_df; backend = :text,
    fit_table_in_display_horizontally = false,
    fit_table_in_display_vertically = false,
    table_format = TextTableFormat(borders = text_table_borders__compact))

> __Interpreting the bootstrap results:__
>
> A bootstrap-to-theoretical SE ratio near 1.0 confirms that the Gaussian error assumption underlying OLS is reasonable for this ticker. If the ratio were much larger than 1.0, it would indicate heavier tails or heteroskedasticity in the residuals, and the theoretical CIs would understate the true uncertainty.

The histograms below show the bootstrap distributions of $\hat{\alpha}$ and $\hat{\beta}$ with the point estimate and 95% confidence interval marked.

In [ ]:
let
    # --- Left panel: bootstrap distribution of α̂ ---
    p1 = histogram(demo_bootstrap["alpha_samples"], bins=50, alpha=0.7, color=:steelblue,
        xlabel="α̂", ylabel="Count", title="Bootstrap Distribution of α̂ ($(demo_ticker))", label="");
    vline!(p1, [est.α], lw=2, color=:red, label="Point estimate");
    vline!(p1, [demo_bootstrap["alpha_ci_95"]...], lw=1.5, ls=:dash, color=:orange, label="95% CI");

    # --- Right panel: bootstrap distribution of β̂ ---
    p2 = histogram(demo_bootstrap["beta_samples"], bins=50, alpha=0.7, color=:steelblue,
        xlabel="β̂", ylabel="Count", title="Bootstrap Distribution of β̂ ($(demo_ticker))", label="");
    vline!(p2, [est.β], lw=2, color=:red, label="Point estimate");
    vline!(p2, [demo_bootstrap["beta_ci_95"]...], lw=1.5, ls=:dash, color=:orange, label="95% CI");

    # --- Combine panels ---
    plot(p1, p2, layout=(1,2), size=(1000, 400), legend=:topright)
end

___
## Task 3: Estimate SIM Parameters for All Tickers and Save
We now run SIM estimation and bootstrap uncertainty quantification for every ticker in the synthetic training dataset and save the complete results to disk.

> __What are we going to do?__
>
> We loop over all 424 tickers, calling `estimate_sim` and `bootstrap_sim` for each. The results are saved to a JLD2 file that downstream sessions (Sessions 2-4) can load directly.

The code below estimates SIM parameters and bootstrap distributions for all tickers.

In [ ]:
# --- Step 1: Allocate storage ---
sim_estimates = MySIMParameterEstimate[];
bootstrap_results = Dict{String, Dict{String,Any}}();

# --- Step 2: Loop over all tickers ---
for (i, ticker) ∈ enumerate(all_tickers)

    # compute daily log growth rates for this ticker
    prices = dataset[ticker][:, :close];
    returns = [(1.0/Δt) * log(prices[t]/prices[t-1]) for t ∈ 2:length(prices)];

    # estimate SIM parameters
    est = estimate_sim(market_returns, returns, ticker; δ=0.0, Δt=Δt);
    push!(sim_estimates, est);

    # bootstrap uncertainty quantification
    bs = bootstrap_sim(market_returns, returns, ticker;
        δ=0.0, Δt=Δt, n_bootstrap=1000, seed=42);
    bootstrap_results[ticker] = bs;

    # progress update every 100 tickers
    if i % 100 == 0
        println("  Processed $(i)/$(length(all_tickers)) tickers...")
    end
end

println("Completed SIM estimation for $(length(sim_estimates)) tickers")

The tables below show the 10 tickers with the highest $R^{2}$ (strongest market-factor exposure) and the 10 with the lowest $R^{2}$ (most idiosyncratic).

In [ ]:
let
    # --- Step 1: Build summary DataFrame ---
    summary_df = DataFrame(
        "Ticker" => [e.ticker for e ∈ sim_estimates],
        "α (ann bps)" => [round(e.α * 252 * 10000, digits=1) for e ∈ sim_estimates],
        "β" => [round(e.β, digits=3) for e ∈ sim_estimates],
        "σ_ε (ann %)" => [round(e.σ_ε * sqrt(252) * 100, digits=1) for e ∈ sim_estimates],
        "R²" => [round(e.r², digits=3) for e ∈ sim_estimates],
        "β CI" => [let bs = bootstrap_results[e.ticker]; ci = bs["beta_ci_95"];
            "[$(round(ci[1], digits=3)), $(round(ci[2], digits=3))]" end for e ∈ sim_estimates]
    );

    # --- Step 2: Sort by R² ---
    sort!(summary_df, :R², rev=true);

    # --- Step 3: Display top 10 ---
    println("Top 10 tickers by R² (strongest market-factor exposure):")
    pretty_table(first(summary_df, 10); backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    # --- Step 4: Display bottom 10 ---
    println("\nBottom 10 tickers by R² (most idiosyncratic):")
    pretty_table(last(summary_df, 10); backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

Save the complete SIM estimation results to disk for use in downstream sessions. The saved file contains the parameter estimates, full bootstrap distributions, and market volatility.

The code below saves the results using [the `save_results(...)` function](../../code/docs/build/session1.html).

In [ ]:
# --- Step 1: Compute market volatility (needed for build_sim_covariance downstream) ---
σ_m = std(market_returns);

# --- Step 2: Save to JLD2 ---
save_path = joinpath(_PATH_TO_DATA, "sim-parameter-estimates.jld2");
save_results(save_path, Dict(
    "sim_estimates"     => sim_estimates,
    "bootstrap_results" => bootstrap_results,
    "market_volatility" => σ_m,
    "tickers"           => all_tickers,
    "n_tickers"         => length(all_tickers)
));

println("Saved SIM parameter estimates to: $(save_path)")
println("  $(length(sim_estimates)) ticker estimates")
println("  $(length(bootstrap_results)) bootstrap distributions (1,000 replicates each)")
println("  Market volatility (annualized): $(round(σ_m * sqrt(252) * 100, digits=1))%")

___
## Summary
This example demonstrated SIM parameter estimation for a single asset, bootstrap uncertainty quantification, and batch estimation across the full 424-ticker synthetic training universe.

> __Key Takeaways:__
>
> * **SIM estimation recovers interpretable factor exposures:** Regularized OLS regression decomposes each asset's returns into a market-driven component (beta) and a firm-specific component (alpha and residual volatility). The regression fit measures how much of the asset's variance the single market factor explains.
> * **Bootstrap quantifies parameter reliability:** Residual resampling produces empirical confidence intervals for alpha and beta without distributional assumptions beyond the fitted model. A bootstrap-to-theoretical SE ratio near 1.0 confirms the Gaussian error model is reasonable.
> * **Batch estimation creates a reusable parameter set:** Running SIM estimation and bootstrap across all 424 tickers produces a complete parameter file that downstream sessions load directly. This avoids re-estimation and ensures consistency across the course.

The saved parameter file is used in the [Sharpe ratio example](eCornell-AI-Finance-S1-Example-SharpeRatio-May-2026.ipynb) and in Sessions 2-4 for preference weight computation and covariance construction.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.